# Exp 4 - Rate Monotonic Scheduling (RMS) using Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Simulate Rate Monotonic Scheduling for periodic real-time tasks.

RMS is a fixed-priority scheduling policy: the shorter the task period, the higher the priority. This makes sense for periodic control loops because faster loops usually need more frequent CPU service.

## Textbook Notes and Case Studies

### 1. Textbook Background

Rate Monotonic Scheduling is a fixed-priority scheduling policy for periodic real-time tasks. The shorter the task period, the higher the priority. RMS is important because many embedded control workloads are periodic: sensor sampling, control updates, actuator checks, and communication heartbeats.

The key design assumption is that task periods and worst-case execution times are known. If these assumptions are weak, the RMS schedulability conclusion is also weak. Real systems must measure or bound worst-case execution time, not only average execution time.

### 2. Architecture Notes

```
Periodic Task Set
  |
  v
Assign Priority by Period
  |
  v
Ready Queue Sorted by Priority
  |
  v
CPU Executes Highest-Priority Ready Task
  |
  v
Deadline / Response-Time Verification
```

Because RMS uses fixed priorities, the scheduler is predictable and simple. This makes it attractive for microcontrollers and embedded kernels. The cost is that some task sets schedulable by dynamic-priority methods may fail under RMS.

### 3. Important Formulas

Task utilization:

```
U_i = C_i / T_i
```

Total utilization:

```
U = sum(C_i / T_i)
```

Liu and Layland RMS sufficient utilization bound for n independent periodic tasks:

```
U <= n * (2^(1/n) - 1)
```

As n becomes large, the bound approaches approximately:

```
U <= 0.693
```

This is a sufficient test, not a necessary test. If a task set passes, it is schedulable under the model. If it fails, it may still be schedulable, so more exact response-time analysis may be needed.

### 4. Classroom Case Studies

Case Study A - Engine Control Unit:
Crankshaft position may need very frequent sampling, while temperature monitoring can run less frequently. RMS naturally gives the crankshaft task higher priority because it has the shorter period.

Case Study B - Autonomous Steering Controller:
A steering loop with a 10 ms period competes with diagnostics at 100 ms. If diagnostics temporarily consume too much CPU, RMS prevents them from blocking the steering loop because diagnostics have lower priority.

Case Study C - Communication Heartbeats:
Safety heartbeat messages may run periodically. RMS analysis helps determine whether adding a new logging task will overload the processor.

### 5. Analysis Checklist

Report each task's period, execution time, priority, utilization, total utilization, and bound. State whether the bound is sufficient or inconclusive. Do not claim failure solely because utilization exceeds the RMS bound.

### 6. Source Notes

- Liu and Layland's original fixed-priority scheduling paper: https://dl.acm.org/doi/10.1145/321738.321743
- Python timing utilities used in simulation: https://docs.python.org/3/library/time.html


## Architecture

```text
Periodic Task Set
  |-- computation time C
  |-- period T
  |-- relative deadline D
          |
          v
RMS Priority Assignment
  |-- shortest period gets highest priority
          |
          v
Timeline Simulation
  |-- release jobs
  |-- run highest-priority ready job
  |-- check deadlines
          |
          v
Schedulability Summary
```

## Formulas and Required Theory

Task utilization:

\[
U_i = \frac{C_i}{T_i}
\]

Total utilization:

\[
U = \sum_{i=1}^{n}\frac{C_i}{T_i}
\]

Liu and Layland sufficient RMS bound:

\[
U \le n(2^{1/n}-1)
\]

This bound is sufficient, not necessary. If a task set is above the bound, simulation or a stronger schedulability test is still needed before declaring it impossible.

## In-Lab Method

1. Define periodic tasks with execution time and period.
2. Compute hyperperiod using least common multiple.
3. Assign fixed priorities by period.
4. Simulate releases over one hyperperiod.
5. Record the timeline and count deadline misses.

In [1]:
from math import gcd
from functools import reduce

def lcm(a, b):
    return a * b // gcd(a, b)

tasks = [
    {"name": "T1 steering", "period": 4, "exec": 1, "deadline": 4},
    {"name": "T2 perception", "period": 5, "exec": 1, "deadline": 5},
    {"name": "T3 planning", "period": 20, "exec": 4, "deadline": 20},
]
hyperperiod = reduce(lcm, [t["period"] for t in tasks])
utilization = sum(t["exec"] / t["period"] for t in tasks)
tasks = sorted(tasks, key=lambda t: t["period"])
ready = []
timeline = []
misses = []
for time in range(hyperperiod):
    for task in tasks:
        if time % task["period"] == 0:
            ready.append({"name": task["name"], "remaining": task["exec"], "deadline": time + task["deadline"]})
    ready.sort(key=lambda job: next(t["period"] for t in tasks if t["name"] == job["name"]))
    if ready:
        job = ready[0]
        timeline.append(job["name"].split()[0])
        job["remaining"] -= 1
        if job["remaining"] == 0:
            ready.pop(0)
    else:
        timeline.append("Idle")
    misses.extend(job for job in ready if time + 1 > job["deadline"])
    ready = [job for job in ready if time + 1 <= job["deadline"]]

print("EXP 4 - IN-LAB RMS RESULT")
print("Utilization:", round(utilization, 3))
print("Hyperperiod:", hyperperiod)
print("Deadline misses:", len(misses))
print("Timeline:", " | ".join(timeline))

EXP 4 - IN-LAB RMS RESULT
Utilization: 0.65
Hyperperiod: 20
Deadline misses: 0
Timeline: T1 | T2 | T3 | T3 | T1 | T2 | T3 | T3 | T1 | Idle | T2 | Idle | T1 | Idle | Idle | T2 | T1 | Idle | Idle | Idle


## Post-Lab Method

The post-lab cell calculates utilization for multiple task sets and compares each against the RMS utilization bound.

In [2]:
def rms_bound(n):
    return n * (2 ** (1 / n) - 1)

task_sets = {
    "Set A": [(1, 4), (1, 5), (4, 20)],
    "Set B": [(1, 3), (1, 4), (2, 6)],
    "Set C": [(2, 5), (2, 7), (4, 12)],
}
print("EXP 4 - POST-LAB UTILIZATION TEST")
print(f"{'Set':6} {'Utilization':>12} {'RMS Bound':>10} {'Bound Test':>12}")
for name, pairs in task_sets.items():
    util = sum(c / p for c, p in pairs)
    bound = rms_bound(len(pairs))
    print(f"{name:6} {util:12.3f} {bound:10.3f} {('Pass' if util <= bound else 'Needs simulation'):>12}")

EXP 4 - POST-LAB UTILIZATION TEST
Set     Utilization  RMS Bound   Bound Test
Set A         0.650      0.780         Pass
Set B         0.917      0.780 Needs simulation
Set C         1.019      0.780 Needs simulation


## What to Write in the Lab Record

- Show each task's period and execution time.
- Show total utilization.
- Show RMS bound for the number of tasks.
- State whether the bound guarantees schedulability or whether simulation is needed.

## References

- Rate-monotonic scheduling utilization formula and Liu-Layland bound: https://en.wikipedia.org/wiki/Rate-monotonic_scheduling
- Python `math` module documentation: https://docs.python.org/3/library/math.html